# Insurance Sales Data Cleaning Pipeline
## پاکسازی داده‌های فروش بیمه

این نوت‌بوک یک دیتاست خام و به‌هم‌ریخته از فروش بیمه (بیش از ۱۰۰۰ ردیف) رو مرحله به مرحله پاک‌سازی می‌کنه.

**مشکلات رایج داده‌ی خام:**
- فرمت‌های مختلف تاریخ
- مقادیر گم‌شده در چند ستون
- ورودی‌های متنی ناهماهنگ (بزرگ/کوچک بودن حروف، غلط تایپی)
- فرمت‌های ناهماهنگ مبلغ حق بیمه (Premium)
- شماره تلفن با فرمت‌های مختلف
- سن‌های نامعتبر (Outlier)
- ردیف‌های تکراری


In [1]:
# Import libraries
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)


## 1. Load and inspect the raw data
### بارگذاری و بررسی اولیه‌ی داده

In [2]:
df = pd.read_csv('../data/raw_insurance_sales.csv')
print("Shape:", df.shape)
df.head()


Shape: (1290, 11)


,policy_id,customer_name,phone,customer_age,policy_type,branch,sale_date,premium_amount,seller_name,status,payment_method
0,POL-10798,Debra Coleman,+989238066223,45,Health,Tehran,2024-09-26,10014043,Charlene Baker,Renewed,Installment
1,POL-10813,Brenda Cantrell,09332432016,32,Travle,isfahan,15/05/2024,1135969,Raymond Walters,Renewed,Cash
2,POL-10746,William Christian,+989274847890,73,Motor,Tehran,06-24-2024,5717885,Julian May,Active,Cash
3,POL-10912,Michael Miller,0988-242-9485,32,Helth,Mashhad,09-18-2023,9077626,Joseph Holmes,Renewed,Installment
4,POL-10568,Vincent Bond,0990-288-9983,62,health,Shiraz,2025-08-21,4627616,Charlene Ramirez,Cancelled,Bank Transfer


In [3]:
# Quick overview of data types and missing values
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1290 entries, 0 to 1289
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   policy_id       1290 non-null   str  
 1   customer_name   1252 non-null   str  
 2   phone           1236 non-null   str  
 3   customer_age    1290 non-null   int64
 4   policy_type     1290 non-null   str  
 5   branch          1290 non-null   str  
 6   sale_date       1290 non-null   str  
 7   premium_amount  1236 non-null   str  
 8   seller_name     1290 non-null   str  
 9   status          1290 non-null   str  
 10  payment_method  1290 non-null   str  
dtypes: int64(1), str(10)
memory usage: 111.0 KB


In [4]:
# Count of missing values per column
df.isna().sum().sort_values(ascending=False)


phone             54
premium_amount    54
customer_name     38
policy_id          0
customer_age       0
branch             0
policy_type        0
sale_date          0
seller_name        0
status             0
payment_method     0
dtype: int64

## 2. Remove exact duplicate rows
### حذف ردیف‌های تکراری کامل

In [5]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows. Remaining: {after}")


Removed 40 duplicate rows. Remaining: 1250


## 3. Clean text columns (policy_type, branch)
### یکسان‌سازی ستون‌های متنی

مشکل: مقادیر مثل `Life`, `life`, `LIFE`, `Life ` همگی باید یک مقدار واحد بشن.


In [6]:
def clean_category(value):
    if pd.isna(value):
        return np.nan
    return str(value).strip().title()

df['policy_type'] = df['policy_type'].apply(clean_category)
df['branch'] = df['branch'].apply(clean_category)

# Fix known typos after standard casing
policy_type_fixes = {
    'Liabilty': 'Liability',
    'Travle': 'Travel',
    'Helth': 'Health',
    'Moter': 'Motor',
}
branch_fixes = {
    'Esfahan': 'Isfahan',
    'Mashad': 'Mashhad',
}
df['policy_type'] = df['policy_type'].replace(policy_type_fixes)
df['branch'] = df['branch'].replace(branch_fixes)

print(df['policy_type'].value_counts())
print(df['branch'].value_counts())


policy_type
Health       240
Fire         216
Life         205
Travel       199
Liability    196
Motor        194
Name: count, dtype: int64
branch
Isfahan    229
Tehran     217
Tabriz     210
Shiraz     208
Mashhad    196
Karaj      190
Name: count, dtype: int64


## 4. Standardize customer_name
### یکسان‌سازی نام مشتری (حذف فاصله‌های اضافه)

In [7]:
df['customer_name'] = df['customer_name'].apply(
    lambda x: x.strip() if isinstance(x, str) else x
)
missing_names = df['customer_name'].isna().sum()
print(f"Missing customer names: {missing_names}")


Missing customer names: 38


## 5. Parse inconsistent date formats
### تبدیل تاریخ‌ها به یک فرمت واحد

فرمت‌های ورودی: `YYYY-MM-DD`, `DD/MM/YYYY`, `MM-DD-YYYY`, `YYYY/MM/DD`


In [8]:
def parse_mixed_date(value):
    if pd.isna(value):
        return pd.NaT
    formats = ['%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%Y/%m/%d']
    for fmt in formats:
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT  # could not parse

df['sale_date'] = df['sale_date'].apply(parse_mixed_date)
print(f"Unparsed dates: {df['sale_date'].isna().sum()}")
df['sale_date'].head()


Unparsed dates: 0


0   2024-09-26
1   2024-05-15
2   2024-06-24
3   2023-09-18
4   2025-08-21
Name: sale_date, dtype: datetime64[us]

## 6. Clean premium_amount (GWP)
### پاکسازی مبلغ حق بیمه

مشکلات: کاما در اعداد (`6,258,777`)، متن اضافه (`تومان`)، مقدار خالی، و مقدار منفی نامعتبر (`-1`).


In [9]:
def clean_premium(value):
    if pd.isna(value) or str(value).strip() == '':
        return np.nan
    text = str(value)
    text = text.replace(',', '').replace('تومان', '').strip()
    try:
        number = float(text)
    except ValueError:
        return np.nan
    if number <= 0:
        return np.nan  # invalid / negative premium treated as missing
    return number

df['premium_amount'] = df['premium_amount'].apply(clean_premium)
print(df['premium_amount'].describe())


count    1.151000e+03
mean     6.834095e+06
std      4.967606e+06
min      1.000000e+05
25%      3.316920e+06
50%      5.448129e+06
75%      8.791421e+06
max      2.689607e+07
Name: premium_amount, dtype: float64


## 7. Clean phone numbers
### یکسان‌سازی شماره تلفن به فرمت 09xxxxxxxxx

In [10]:
def clean_phone(value):
    if pd.isna(value):
        return np.nan
    digits = re.sub(r'\D', '', str(value))  # keep digits only
    # Normalize country code variants to local 0-prefixed format
    if digits.startswith('98') and len(digits) == 12:
        digits = '0' + digits[2:]
    elif digits.startswith('0098') and len(digits) == 14:
        digits = '0' + digits[4:]
    if len(digits) == 11 and digits.startswith('09'):
        return digits
    return np.nan  # invalid phone number

df['phone'] = df['phone'].apply(clean_phone)
print(f"Invalid/missing phones after cleaning: {df['phone'].isna().sum()}")


Invalid/missing phones after cleaning: 53


## 8. Handle invalid customer_age values
### اصلاح سن‌های نامعتبر (Outlier)

In [11]:
# Realistic age range for insurance customers: 18 to 100
invalid_age_mask = ~df['customer_age'].between(18, 100)
print(f"Invalid ages found: {invalid_age_mask.sum()}")
df.loc[invalid_age_mask, 'customer_age'] = np.nan


Invalid ages found: 27


## 9. Handle remaining missing values
### تصمیم‌گیری نهایی درباره‌ی مقادیر گم‌شده

استراتژی:
- `customer_name` / `phone` گم‌شده: نگه داشته میشه (چون حذف کامل ردیف اطلاعات فروش رو از بین می‌بره) ولی flag میشه.
- `premium_amount` گم‌شده: با میانه‌ی (median) همون نوع بیمه پر میشه.
- `sale_date` گم‌شده/نامعتبر: ردیف حذف میشه چون تاریخ فروش برای تحلیل حیاتیه.


In [12]:
# Drop rows where sale_date could not be parsed (critical field)
before = len(df)
df = df.dropna(subset=['sale_date'])
print(f"Dropped {before - len(df)} rows with invalid sale_date")

# Fill missing premium_amount with the median premium for that policy_type
df['premium_amount'] = df.groupby('policy_type')['premium_amount'].transform(
    lambda x: x.fillna(x.median())
)

# Flag rows with missing contact info instead of dropping them
df['missing_contact_info'] = df['customer_name'].isna() | df['phone'].isna()


Dropped 0 rows with invalid sale_date


## 10. Final validation
### بررسی نهایی کیفیت داده

In [13]:
print("Final shape:", df.shape)
print()
print("Remaining missing values per column:")
print(df.isna().sum())
print()
print("Data types:")
print(df.dtypes)


Final shape: (1250, 12)

Remaining missing values per column:
policy_id                0
customer_name           38
phone                   53
customer_age            27
policy_type              0
branch                   0
sale_date                0
premium_amount           0
seller_name              0
status                   0
payment_method           0
missing_contact_info     0
dtype: int64

Data types:
policy_id                          str
customer_name                      str
phone                              str
customer_age                   float64
policy_type                        str
branch                             str
sale_date               datetime64[us]
premium_amount                 float64
seller_name                        str
status                             str
payment_method                     str
missing_contact_info              bool
dtype: object


In [14]:
df.describe(include='all')


,policy_id,customer_name,phone,customer_age,policy_type,branch,sale_date,premium_amount,seller_name,status,payment_method,missing_contact_info
count,1250,1212,1197,1223.000000,1250,1250,1250,1.250000e+03,1250,1250,1250,1250
unique,1250,1195,1197,NaN,6,6,NaN,NaN,25,4,4,2
top,POL-10798,Paul Williams,09238066223,NaN,Health,Isfahan,NaN,NaN,Raymond Walters,Cancelled,Credit Card,False
freq,1,2,1,NaN,240,229,NaN,NaN,58,326,340,1162
mean,NaN,NaN,NaN,46.154538,NaN,NaN,2025-01-13 19:22:22.080000,6.880230e+06,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,18.000000,NaN,NaN,2023-07-07 00:00:00,1.000000e+05,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,32.000000,NaN,NaN,2024-04-29 00:00:00,3.370608e+06,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,46.000000,NaN,NaN,2025-01-08 12:00:00,5.491877e+06,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,61.000000,NaN,NaN,2025-09-30 18:00:00,8.709218e+06,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,74.000000,NaN,NaN,2026-07-06 00:00:00,2.689607e+07,NaN,NaN,NaN,NaN


## 11. Save the cleaned dataset
### ذخیره‌ی داده‌ی پاک‌شده

In [15]:
df.to_csv('../data/cleaned_insurance_sales.csv', index=False, encoding='utf-8-sig')
print("Saved cleaned dataset to data/cleaned_insurance_sales.csv")


Saved cleaned dataset to data/cleaned_insurance_sales.csv


## Summary / خلاصه

| Step | Issue Found | Action Taken |
|---|---|---|
| Duplicates | ~40 exact duplicate rows | Dropped |
| Categorical text | Inconsistent casing & typos (policy_type, branch) | Standardized via `.title()` + typo fix map |
| Dates | 4 different date formats | Parsed into a single `datetime` column |
| Premium amount | Commas, currency text, blanks, negative values | Parsed to numeric, invalid values treated as missing, then median-imputed per policy type |
| Phone numbers | Multiple formats (+98, 0098, dashes) | Normalized to local `09xxxxxxxxx` format |
| Customer age | A few unrealistic values (e.g. -5, 150) | Treated as missing |
| Missing sale_date | Could not be parsed | Rows dropped (critical field) |

این پایپ‌لاین رو می‌شه روی هر دیتاست فروش مشابهی (با تغییرات کوچیک) دوباره استفاده کرد.
